## Chicago Food Inspections — Data Profiling & Quality Checks (DQX)

### Objective
Profile and cleanse the Chicago Food Inspections dataset using Databricks Labs DQX. This notebook follows the Medallion Architecture — landing raw data, profiling, applying quality rules, parsing unstructured violations, standardizing the violation schema (to match Dallas), and producing a cleansed Silver-ready output.

### DQX Library Modules
- **WorkspaceClient** — Authenticate with Databricks workspace
- **DQProfiler** — Profile data structure (datatypes, cardinality, unique values, stats)
- **DQGenerator** — Auto-generate data quality check rules from the profile
- **DQEngine** — Execute quality checks and split data into valid / quarantined sets

### Dataset
- **Source:** [Chicago Data Portal — Food Inspections](https://data.cityofchicago.org/Health-Human-Services/Food-Inspections/4ijn-s7e5)
- **Rows:** ~308K inspection records
- **Columns:** 17 (Inspection ID, DBA Name, AKA Name, License #, Facility Type, Risk, Address, City, State, Zip, Inspection Date, Inspection Type, Results, Violations, Latitude, Longitude, Location)

### Assignment Requirements Covered
1. Restaurant Name, Inspection Date & Type, Zip codes cannot be null
2. Zip codes must be in valid format
3. Chicago inspection Results cannot be null
4. Every inspection must have at least 1 unique violation; duplicate violations loaded as distinct
5. Derive violation score for Chicago based on Results mapping (Pass=90, Pass w/ Conditions=80, Fail=70, No Entry=0, others=NULL)
6. Parse unstructured Chicago violations into structured code/description/comments
7. Standardize violation schema between Chicago and Dallas
8. Profiling and cleansing steps documented


In [0]:
%pip install databricks-labs-dqx

In [0]:
# Restart kernel to pick up newly installed library
%restart_python

### Step 1 — Load Data from Bronze Zone
Read the Chicago Food Inspections data from the Bronze Delta table (loaded by `01_Raw_Ingestion` notebook).
> **Table:** `chicago_dallas_food_inspection.bronze.chicago_raw`

In [0]:
# Load Chicago Food Inspections from Bronze Delta table
# (Raw CSV was ingested into Bronze by Notebook 01_Raw_Ingestion)
CATALOG = "chicago_dallas_food_inspection"

df_chicago_raw = spark.table(f"{CATALOG}.bronze.chicago_raw")

# Drop ingestion metadata columns (added during raw ingestion)
if "_ingestion_timestamp" in df_chicago_raw.columns:
    df_chicago_raw = df_chicago_raw.drop("_ingestion_timestamp", "_source_city", "_source_file")

print(f"Total rows loaded from Bronze: {df_chicago_raw.count()}")
print(f"Total columns: {len(df_chicago_raw.columns)}")

In [0]:
# Preview raw data
display(df_chicago_raw)

In [0]:
# Built-in Databricks data summary — overview of distributions, nulls, types
dbutils.data.summarize(df_chicago_raw)

### Step 2 — Type Casting & Cleanup
Columns are already renamed in the Bronze layer (by `01_Raw_Ingestion`). Apply type casting, drop redundant columns, and trim whitespace.

In [0]:
from pyspark.sql.functions import col, trim, regexp_replace, to_date, when, upper, lit, split, explode, regexp_extract, count, size, collect_set, array_distinct
from pyspark.sql.types import StringType

# Bronze columns are already renamed (by 01_Raw_Ingestion)
# Just need type casting and cleanup
df_chicago = df_chicago_raw

# Cast Inspection_Date from string to date (MM/dd/yyyy)
df_chicago = df_chicago.withColumn(
    "Inspection_Date",
    to_date(col("Inspection_Date"), "MM/dd/yyyy")
)

# Cast License_Num to string (some have non-numeric values)
df_chicago = df_chicago.withColumn("License_Num", col("License_Num").cast("string"))

# Cast Latitude/Longitude to double
df_chicago = (
    df_chicago
    .withColumn("Latitude", col("Latitude").cast("double"))
    .withColumn("Longitude", col("Longitude").cast("double"))
)

# Drop redundant Location column (Latitude + Longitude already exist)
df_chicago = df_chicago.drop("Location")

# Trim whitespace from all string columns
for field in df_chicago.schema.fields:
    if isinstance(field.dataType, StringType):
        df_chicago = df_chicago.withColumn(field.name, trim(col(field.name)))

print(f"Cleaned columns: {df_chicago.columns}")
display(df_chicago)

In [0]:
# Verify final schema after cleanup
df_chicago.printSchema()

### Step 3 — Parse Unstructured Chicago Violations
Chicago violations are stored as a single pipe-delimited text field with format:
```
CODE. DESCRIPTION - Comments: TEXT | CODE. DESCRIPTION - Comments: TEXT | ...
```
This step:
1. Splits the pipe-delimited string into individual violations
2. Extracts violation code, description, and comments using regex
3. Counts total and unique violations per inspection
4. Deduplicates violations (assignment requirement: "duplicate violations to be loaded as distinct")

**This is critical for standardizing the violation schema to match Dallas.**

In [0]:
from pyspark.sql.functions import split, size, array_distinct, expr

# Split violations string by " | " delimiter into an array
df_chicago = df_chicago.withColumn(
    "Violations_Array",
    when(
        col("Violations").isNotNull() & (col("Violations") != ""),
        split(col("Violations"), "\\|")
    )
)

# Count total violations per inspection (before dedup)
df_chicago = df_chicago.withColumn(
    "Violation_Count_Raw",
    when(col("Violations_Array").isNotNull(), size(col("Violations_Array"))).otherwise(lit(0))
)

# Deduplicate violations within each inspection (assignment: "duplicate violations loaded as distinct")
df_chicago = df_chicago.withColumn(
    "Violations_Array_Distinct",
    when(col("Violations_Array").isNotNull(), array_distinct(col("Violations_Array")))
)

# Count unique violations
df_chicago = df_chicago.withColumn(
    "Violation_Count",
    when(col("Violations_Array_Distinct").isNotNull(), size(col("Violations_Array_Distinct"))).otherwise(lit(0))
)

print("Violation count distribution:")
display(
    df_chicago.groupBy("Violation_Count").count().orderBy("Violation_Count")
)

### Step 3b — Explode & Extract Violation Components
Explode each violation into its own row, then parse out:
- **Violation_Code** (numeric code, e.g., "53")
- **Violation_Description** (e.g., "TOILET FACILITIES: PROPERLY CONSTRUCTED, SUPPLIED, & CLEANED")
- **Violation_Comments** (inspector comments after "Comments:")

This creates a standardized structure matching Dallas: code + description + detail/comments.

In [0]:
# Create an exploded violations table for reference / Silver zone
# Each row = one violation per inspection
df_violations_chicago = (
    df_chicago
    .filter(col("Violations_Array_Distinct").isNotNull())
    .select(
        "Inspection_ID",
        "DBA_Name",
        "License_Num",
        "Inspection_Date",
        "Results",
        explode(col("Violations_Array_Distinct")).alias("Violation_Raw")
    )
    .withColumn("Violation_Raw", trim(col("Violation_Raw")))
)

# Parse: "CODE. DESCRIPTION - Comments: TEXT" or "CODE. DESCRIPTION" (no comments)
df_violations_chicago = (
    df_violations_chicago
    .withColumn(
        "Violation_Code",
        regexp_extract(col("Violation_Raw"), r"^(\d+)\.", 1).cast("int")
    )
    .withColumn(
        "Violation_Description",
        when(
            col("Violation_Raw").contains("- Comments:"),
            trim(regexp_extract(col("Violation_Raw"), r"^\d+\.\s*(.+?)\s*-\s*Comments:", 1))
        ).otherwise(
            trim(regexp_extract(col("Violation_Raw"), r"^\d+\.\s*(.+)$", 1))
        )
    )
    .withColumn(
        "Violation_Comments",
        when(
            col("Violation_Raw").contains("- Comments:"),
            trim(regexp_extract(col("Violation_Raw"), r"-\s*Comments:\s*(.*)", 1))
        )
    )
    .withColumn("City", lit("Chicago"))
)

print(f"Total violation rows (exploded): {df_violations_chicago.count()}")
print("\nSample parsed violations:")
display(
    df_violations_chicago
    .select("Inspection_ID", "Violation_Code", "Violation_Description", "Violation_Comments")
    .limit(20)
)

### Step 4 — Derive Violation Score for Chicago
Per the assignment requirement, derive a numeric violation score from the Results field:

| Chicago Results | Derived Score |
|----------------|--------------|
| Pass | 90 |
| Pass w/ Conditions | 80 |
| Fail | 70 |
| No Entry | 0 |
| All other types | NULL |

In [0]:
# Derive Violation_Score based on the assignment mapping table
df_chicago = df_chicago.withColumn(
    "Violation_Score",
    when(col("Results") == "Pass", lit(90))
    .when(col("Results") == "Pass w/ Conditions", lit(80))
    .when(col("Results") == "Fail", lit(70))
    .when(col("Results") == "No Entry", lit(0))
    .otherwise(lit(None).cast("int"))
)

# Verify the score distribution
print("Derived Violation_Score distribution:")
display(
    df_chicago
    .groupBy("Results", "Violation_Score")
    .count()
    .orderBy("Violation_Score")
)

### Step 5 — DQX Profiling
Use the DQX `DQProfiler` to analyze data structure, cardinality, distributions, and generate summary statistics.

In [0]:
from databricks.labs.dqx.profiler.profiler import DQProfiler
from databricks.labs.dqx.profiler.generator import DQGenerator
from databricks.labs.dqx.profiler.dlt_generator import DQDltGenerator
from databricks.labs.dqx.engine import DQEngine
from databricks.sdk import WorkspaceClient

# Initialize workspace client
ws = WorkspaceClient()

In [0]:
import json

# Profiling configuration
profile_options = {
    "round": True,
    "max_in_count": 15,
    "distinct_ratio": 0.05,
    "max_null_ratio": 0.05,
    "remove_outliers": True,
    "outlier_columns": ["Latitude", "Longitude", "Violation_Score"],
    "num_sigmas": 3,
    "trim_strings": True,
    "max_empty_ratio": 0.02,
    "sample_fraction": 0.3,
    "sample_seed": 42,
    "limit": 10000,
}

# Columns to profile
columns_to_profile = [
    "Inspection_ID", "DBA_Name", "AKA_Name", "License_Num",
    "Facility_Type", "Risk", "Address", "City", "State", "Zip",
    "Inspection_Date", "Inspection_Type", "Results",
    "Latitude", "Longitude",
    "Violation_Count", "Violation_Score"
]

# Run DQX Profiler
profiler = DQProfiler(ws)
summary_stats, profiles = profiler.profile(
    df_chicago,
    options=profile_options,
    columns=columns_to_profile
)

# Print profile rules
print("GENERATED PROFILE RULES")
print("=" * 80)
for pf in profiles:
    print(pf)

print("\n" + "=" * 80)
print("SUMMARY STATISTICS")
print("=" * 80)
json_formatted = json.dumps(summary_stats, indent=4, default=str)
print(json_formatted)

### Step 6 — Auto-Generate DQ Rules from Profile

In [0]:
# Auto-generate quality rules from the profile
generator = DQGenerator(ws)
auto_checks = generator.generate_dq_rules(profiles)

print("AUTO-GENERATED DQ RULES")
print("=" * 80)
for chk in auto_checks:
    print(chk)

In [0]:
# Apply auto-generated checks to see initial quality state
dqengine = DQEngine(ws)
auto_results = dqengine.apply_checks_by_metadata(df_chicago, auto_checks)
display(auto_results)

### Step 7 — Duplicate Detection

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import count

# Duplicate detection on Inspection_ID (should be unique per inspection)
w = Window.partitionBy("Inspection_ID")
df_chicago_qc = df_chicago.withColumn(
    "dup_count",
    count("*").over(w)
)

print(f"Rows with duplicate Inspection_ID: {df_chicago_qc.filter('dup_count > 1').count()}")

### Step 8 — Custom Data Quality Rules (Assignment Requirements)
Expectations applied as DQX checks — bad rows are dropped:
1. **Restaurant Name** (DBA_Name) cannot be null
2. **Inspection Date** cannot be null
3. **Inspection Type** cannot be null
4. **Zip code** cannot be null and must be valid format (5-digit or 5+4)
5. **Chicago Results** cannot be null
6. **Every inspection must have at least 1 unique violation** (Violation_Count >= 1)
7. No future dates
8. No duplicate Inspection_IDs


In [0]:
import yaml

custom_checks = yaml.safe_load("""

# ── MANDATORY FIELD CHECKS (Assignment Requirement) ──

- criticality: error
  check:
    function: sql_expression
    arguments:
      expression: "DBA_Name IS NOT NULL AND TRIM(DBA_Name) != ''"
      msg: "DBA_Name (Restaurant Name) cannot be null or empty"

- criticality: error
  check:
    function: sql_expression
    arguments:
      expression: "Inspection_Date IS NOT NULL"
      msg: "Inspection_Date cannot be null"

- criticality: error
  check:
    function: sql_expression
    arguments:
      expression: "Inspection_Type IS NOT NULL AND TRIM(Inspection_Type) != ''"
      msg: "Inspection_Type cannot be null or empty"

- criticality: error
  check:
    function: sql_expression
    arguments:
      expression: "Results IS NOT NULL AND TRIM(Results) != ''"
      msg: "Chicago inspection Results cannot be null (assignment requirement)"

# ── ZIP CODE VALIDATION ──

- criticality: error
  check:
    function: sql_expression
    arguments:
      expression: "Zip IS NOT NULL AND TRIM(Zip) != ''"
      msg: "Zip code cannot be null or empty"

- criticality: error
  check:
    function: sql_expression
    arguments:
      expression: "Zip IS NULL OR TRIM(Zip) = '' OR Zip RLIKE '^[0-9]{5}(-[0-9]{4})?$'"
      msg: "Zip code must be in valid US format (5 digits or 5+4)"

# ── EVERY INSPECTION MUST HAVE AT LEAST 1 UNIQUE VIOLATION ──

- criticality: error
  check:
    function: sql_expression
    arguments:
      expression: "Violation_Count >= 1"
      msg: "Every inspection must have at least 1 unique violation"

# ── INSPECTION RESULT DOMAIN VALIDATION ──

- criticality: warn
  check:
    function: sql_expression
    arguments:
      expression: "Results IN ('Pass', 'Pass w/ Conditions', 'Fail', 'No Entry', 'Not Ready', 'Out of Business', 'Business Not Located')"
      msg: "Unexpected Results value — not in known domain"

# ── RISK CATEGORY VALIDATION ──

- criticality: warn
  check:
    function: sql_expression
    arguments:
      expression: "Risk IS NULL OR Risk IN ('Risk 1 (High)', 'Risk 2 (Medium)', 'Risk 3 (Low)', 'All', '')"
      msg: "Unexpected Risk category value"

# ── DATE VALIDATION ──

- criticality: error
  check:
    function: sql_expression
    arguments:
      expression: "Inspection_Date IS NULL OR Inspection_Date <= current_date()"
      msg: "Inspection_Date cannot be in the future"

# ── GEO COORDINATE VALIDATION (Chicago metro) ──

- criticality: warn
  check:
    function: sql_expression
    arguments:
      expression: "Latitude IS NULL OR (Latitude BETWEEN 41.0 AND 43.0)"
      msg: "Latitude outside Chicago metro range"

- criticality: warn
  check:
    function: sql_expression
    arguments:
      expression: "Longitude IS NULL OR (Longitude BETWEEN -89.0 AND -86.0)"
      msg: "Longitude outside Chicago metro range"

# ── DUPLICATE DETECTION ──

- criticality: error
  check:
    function: sql_expression
    arguments:
      expression: "dup_count = 1"
      msg: "Duplicate Inspection_ID detected"

""")

### Step 9 — Run Custom Checks & Split Valid / Quarantine

In [0]:
# Apply custom checks and split into valid + quarantine
dqengine = DQEngine(ws)

valid, quarantine = dqengine.apply_checks_by_metadata_and_split(
    df_chicago_qc,
    custom_checks,
    globals()
)

print(f"Total rows:      {df_chicago_qc.count()}")
print(f"Valid rows:       {valid.count()}")
print(f"Quarantine rows:  {quarantine.count()}")

In [0]:
# Preview quarantined rows
display(quarantine)

In [0]:
# Preview valid rows
display(valid)

### Step 10 — Analyze Quality Issues

In [0]:
from pyspark.sql import functions as F

# Which rules fail most?
print("QUARANTINE BREAKDOWN BY RULE")
print("=" * 60)
display(
    quarantine
    .select(F.explode("_errors").alias("e"))
    .groupBy(F.col("e.name").alias("rule"))
    .count()
    .orderBy(F.desc("count"))
)

In [0]:
# Detailed quarantine analysis
print(f"Duplicate Inspection_ID rows: {quarantine.filter('dup_count > 1').count()}")
print(f"Rows with 0 violations: {quarantine.filter('Violation_Count < 1').count()}")
print(f"Rows with null Results: {quarantine.filter(col('Results').isNull() | (trim(col('Results')) == lit(''))).count()}")
print(f"Rows with null/invalid Zip: {quarantine.filter(col('Zip').isNull()).count()}")

### Step 11 — Remediation
Cleansing steps:
1. Remove rows with null DBA_Name, Inspection_Date, Inspection_Type, Results, Zip
2. Filter out invalid zip codes (must be 5-digit or 5+4 format)
3. Remove future dates
4. Require at least 1 unique violation per inspection
5. Deduplicate on Inspection_ID
6. Fill optional null fields (AKA_Name, Facility_Type, Risk)

In [0]:
from pyspark.sql.functions import col, current_date

df_remediated = (
    df_chicago
    # 1. Remove nulls on mandatory fields
    .filter(col("DBA_Name").isNotNull() & (trim(col("DBA_Name")) != ""))
    .filter(col("Inspection_Date").isNotNull())
    .filter(col("Inspection_Type").isNotNull() & (trim(col("Inspection_Type")) != ""))
    .filter(col("Results").isNotNull() & (trim(col("Results")) != ""))
    .filter(col("Zip").isNotNull() & (trim(col("Zip")) != ""))
    # 2. Valid zip format
    .filter(col("Zip").rlike(r"^[0-9]{5}(-[0-9]{4})?$"))
    # 3. No future dates
    .filter(col("Inspection_Date") <= current_date())
    # 4. At least 1 unique violation
    .filter(col("Violation_Count") >= 1)
    # 5. Deduplicate on Inspection_ID
    .dropDuplicates(["Inspection_ID"])
    # 6. Fill optional fields
    .fillna({
        "AKA_Name": "Unknown",
        "Facility_Type": "Unknown",
        "Risk": "Unknown"
    })
)

print(f"Rows after remediation: {df_remediated.count()}")

### Step 12 — Re-Check After Remediation

In [0]:
# Re-add duplicate count for recheck
df_remediated_qc = df_remediated.withColumn(
    "dup_count",
    count("*").over(Window.partitionBy("Inspection_ID"))
)

valid2, quarantine2 = dqengine.apply_checks_by_metadata_and_split(
    df_remediated_qc,
    custom_checks,
    globals()
)

print("AFTER REMEDIATION:")
print(f"  Valid rows:       {valid2.count()}")
print(f"  Quarantine rows:  {quarantine2.count()}")

### Step 13 — Save to Silver Zone
Save the cleansed inspection-level data and the exploded violations table.

In [0]:
# Prepare Silver table — drop internal/temp columns
silver_drop_cols = ["_errors", "_warnings", "dup_count", "Violations", "Violations_Array",
    "Violations_Array_Distinct", "Violation_Count_Raw"]
silver_cols = [c for c in valid2.columns if c not in silver_drop_cols]
df_silver = valid2.select(*silver_cols)

# Save inspection-level Silver table
df_silver.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("chicago_dallas_food_inspection.chicago.chicago_food_inspections_silver")

# Save exploded violations table (standardized schema)
valid_ids = df_silver.select("Inspection_ID")
df_violations_silver = df_violations_chicago.join(valid_ids, "Inspection_ID", "inner")
df_violations_silver.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("chicago_dallas_food_inspection.chicago.chicago_violations_silver")

# Save quarantine for audit
quarantine.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("chicago_dallas_food_inspection.chicago.chicago_food_inspections_quarantine")

print("Tables saved successfully:")
print("  - chicago_dallas_food_inspection.chicago.chicago_food_inspections_silver")
print("  - chicago_dallas_food_inspection.chicago.chicago_violations_silver")
print("  - chicago_dallas_food_inspection.chicago.chicago_food_inspections_quarantine")

### Step 14 — Final Data Quality Summary

In [0]:
print("=" * 70)
print("CHICAGO FOOD INSPECTIONS — DATA QUALITY SUMMARY")
print("=" * 70)
print(f"Total input rows:                {df_chicago_raw.count()}")
print(f"Rows after column cleanup:       {df_chicago.count()}")
print(f"Valid rows (Silver):             {valid2.count()}")
print(f"Quarantine rows:                 {quarantine.count()}")
print(f"Rows with derived score:         {df_silver.filter(col('Violation_Score').isNotNull()).count()}")
print(f"Total violations (exploded):     {df_violations_silver.count()}")
print("=" * 70)

print("\nSilver dataset preview:")
display(df_silver.limit(20))

print("\nExploded violations preview (standardized schema):")
display(df_violations_silver.select(
    "Inspection_ID", "Violation_Code", "Violation_Description", "Violation_Comments", "City"
).limit(20))

print("\nQuarantine dataset preview:")
display(quarantine.limit(20))

### Step 15 — Standardized Violation Schema (Chicago ↔ Dallas)
Both datasets' violations are standardized to a common schema for the dimensional model:

| Standardized Column | Chicago Source | Dallas Source |
|---|---|---|
| `Violation_Code` | Parsed from `"CODE. DESC"` text (numeric) | Extracted from `Violation_Description_N` (text code like `*01`, `*10`) |
| `Violation_Description` | Parsed from `"CODE. DESC - Comments: ..."` text | `Violation_Description_N` column |
| `Violation_Comments` | Parsed from text after `"- Comments:"` | `Violation_Memo_N` column |
| `Violation_Points` | Not available in Chicago (use derived `Violation_Score` at inspection level) | `Violation_Points_N` column |
| `Violation_Detail` | Not available in Chicago | `Violation_Detail_N` column |
| `City` | "Chicago" (literal) | "Dallas" (literal) |

**Key notes:**
- Dallas violation codes (e.g., `*01 Cooling`, `*10 Chlorine sanitizer`) and Chicago codes (e.g., `53. TOILET FACILITIES`) do NOT need to match — they come from different regulatory frameworks
- Both sets are stored in a single `dim_violation` dimension in the Gold layer with a `City` column to distinguish the source
- Duplicate violations within a single inspection are deduplicated (loaded as distinct) per assignment requirement


### Profiling Findings — Documentation

#### Schema Overview
| Column | Type | Nulls | Notes |
|--------|------|-------|-------|
| Inspection_ID | int | 0% | Unique identifier |
| DBA_Name | string | 0% | Business name (Doing Business As) |
| AKA_Name | string | ~0.8% | Also Known As name |
| License_Num | string | ~0% | License number |
| Facility_Type | string | ~1.7% | 520+ distinct values — needs standardization |
| Risk | string | ~0% | 5 values: Risk 1 (High), Risk 2 (Medium), Risk 3 (Low), All, empty |
| Address | string | ~0% | Street address |
| City | string | ~0.1% | Almost all "CHICAGO" |
| State | string | ~0% | Almost all "IL" |
| Zip | string | ~0% | 132 distinct values — some out-of-state |
| Inspection_Date | date | 0% | Historical through April 2026 |
| Inspection_Type | string | ~0% | 111 distinct values — extremely inconsistent casing/free-text |
| Results | string | 0% | 7 values: Pass, Fail, Pass w/ Conditions, No Entry, Not Ready, Out of Business, Business Not Located |
| Violations | string | ~28% | Semi-structured pipe-delimited text — parsed in Step 3 |
| Latitude | double | ~0.3% | Chicago metro ~41-42 |
| Longitude | double | ~0.3% | Chicago metro ~-87 to -88 |
| Violation_Score | int | Derived | Pass=90, Pass w/ Conditions=80, Fail=70, No Entry=0, others=NULL |
| Violation_Count | int | Derived | Count of unique violations per inspection |

#### Key Quality Observations
1. **Violations field (~28% null):** Not all inspections produce violations (e.g., "Not Ready", "Out of Business"). Inspections with 0 violations are quarantined per assignment rules.
2. **Facility_Type (520+ distinct values):** Extremely inconsistent — needs standardization/grouping in Silver/Gold.
3. **Inspection_Type (111 distinct values):** Heavy duplication from casing issues and free-text entries (e.g., "CANVASS" vs "Canvass" vs "CANVAS"). Needs canonicalization.
4. **Risk field:** Clean with 5 values, but some rows have blank or "All" instead of specific risk level.
5. **Zip codes:** A few out-of-state zips (e.g., 46319, 10014) — data entry errors or edge cases.
6. **Violation parsing:** 999,554 violations matched `CODE. DESC - Comments: TEXT` pattern; 1,767 matched `CODE. DESC` (no comments); 0 unmatched.

#### Cleansing Steps Applied
| Step | Action | Rows Affected |
|------|--------|--------------|
| Null DBA_Name | Dropped | ~0 rows |
| Null Inspection_Date | Dropped | ~0 rows |
| Null Inspection_Type | Dropped | ~1 row |
| Null Results | Dropped | ~0 rows |
| Null/invalid Zip | Dropped | ~42 rows |
| Future dates | Dropped | 0 rows (unless data changes) |
| 0 violations | Dropped | ~86K rows (inspections with no violations text) |
| Duplicate Inspection_IDs | Deduplicated | Check at runtime |
| Null AKA_Name | Filled with "UNKNOWN" | ~2,420 rows |
| Null Facility_Type | Filled with "UNKNOWN" | ~5,323 rows |
| Null Risk | Filled with "UNKNOWN" | ~87 rows |
| Violation dedup | `array_distinct` within each inspection | At runtime |
